# Parse RMMS Traffic Data

Reads the `N*.traffic.htm` files from `../data/data_raw/` and combines them into a single
`../data/data_processed/traffic_aadt.csv` with truck AADT per road link.

**Table structure in each .htm file:**
- Table index 2 is the main data table (114 rows for N1)
- Rows 0–5 are header rows; data starts at row 6
- Column mapping:
  - 0: Link no (e.g. N1-1L)
  - 1: Name
  - 4: Start Chainage (km)
  - 7: End Chainage (km)
  - 8: Length (km)
  - 9: Heavy Truck AADT
  - 10: Medium Truck AADT
  - 11: Small Truck AADT
  - 24: Total AADT

Note: each chainage has an L (left) and R (right) directional row — we average them per link.

In [ ]:
import pandas as pd
import os

# Roads to parse — the same roads in our simulation network
ROADS = ['N1', 'N2', 'N102', 'N104', 'N105', 'N106', 'N204', 'N207']
DATA_RAW = '../data/data_raw'
DATA_OUT = '../data/data_processed/traffic_aadt.csv'

In [ ]:
def parse_traffic_htm(filepath, road_name):
    """
    Parse one N*.traffic.htm file and return a cleaned DataFrame.
    """
    tables = pd.read_html(filepath)
    
    # Table 2 is the main data table
    t = tables[2]
    
    # Data rows start at index 6 (rows 0-5 are header/title rows)
    data = t.iloc[6:].copy()
    data.columns = range(t.shape[1])
    
    # Select and rename the columns we care about
    df = pd.DataFrame({
        'road': road_name,
        'link_no': data[0],
        'link_name': data[1],
        'start_chainage_km': pd.to_numeric(data[4], errors='coerce'),
        'end_chainage_km': pd.to_numeric(data[7], errors='coerce'),
        'length_km': pd.to_numeric(data[8], errors='coerce'),
        'heavy_truck': pd.to_numeric(data[9], errors='coerce'),
        'medium_truck': pd.to_numeric(data[10], errors='coerce'),
        'small_truck': pd.to_numeric(data[11], errors='coerce'),
        'total_AADT': pd.to_numeric(data[24], errors='coerce'),
    })
    
    # Drop rows where link_no is NaN or doesn't look like a link ID
    df = df.dropna(subset=['link_no', 'heavy_truck'])
    df = df[df['link_no'].astype(str).str.match(r'^[A-Z]')]
    
    # Each chainage has L (left) and R (right) directional rows.
    # Group only by base_link (drop trailing L/R), average traffic values across directions.
    df['base_link'] = df['link_no'].astype(str).str.replace(r'[LR]$', '', regex=True)
    df = df.groupby(['road', 'base_link'], as_index=False).agg({
        'link_name': 'first',
        'start_chainage_km': 'first',
        'end_chainage_km': 'first',
        'length_km': 'first',
        'heavy_truck': 'mean',
        'medium_truck': 'mean',
        'small_truck': 'mean',
        'total_AADT': 'mean',
    })
    
    # Add a combined truck AADT column
    df['truck_AADT'] = df['heavy_truck'] + df['medium_truck'] + df['small_truck']
    
    return df

In [ ]:
# Parse all roads and combine
all_dfs = []

for road in ROADS:
    filepath = os.path.join(DATA_RAW, f'{road}.traffic.htm')
    if not os.path.exists(filepath):
        print(f'WARNING: {filepath} not found, skipping')
        continue
    df = parse_traffic_htm(filepath, road)
    all_dfs.append(df)
    print(f'{road}: {len(df)} links parsed')

traffic_df = pd.concat(all_dfs, ignore_index=True)
print(f'\nTotal links across all roads: {len(traffic_df)}')
traffic_df.head()

In [ ]:
# Quick summary: truck AADT per road
summary = traffic_df.groupby('road').agg(
    links=('base_link', 'count'),
    avg_heavy_truck=('heavy_truck', 'mean'),
    avg_truck_AADT=('truck_AADT', 'mean'),
    max_truck_AADT=('truck_AADT', 'max'),
).round(0)
print(summary.to_string())

In [ ]:
import matplotlib.pyplot as plt

# Bar chart: average truck AADT per road
fig, ax = plt.subplots(figsize=(9, 4))
summary['avg_truck_AADT'].sort_values(ascending=False).plot.bar(ax=ax, color='steelblue')
ax.set_title('Average Truck AADT per Road (Heavy + Medium + Small)')
ax.set_xlabel('Road')
ax.set_ylabel('Average Truck AADT')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig('../data/data_processed/aadt_per_road.png', dpi=150)
plt.show()
print('Saved aadt_per_road.png')

In [ ]:
# Save combined CSV
os.makedirs(os.path.dirname(DATA_OUT), exist_ok=True)
traffic_df.to_csv(DATA_OUT, index=False)
print(f'Saved to {DATA_OUT}')
print(traffic_df.dtypes)